In [2]:
!pip install -q transformers datasets peft accelerate evaluate

In [3]:
!pip install -q "transformers>=4.40.0"

In [4]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np
import torch

In [5]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## Tokenizer & tokenization

In [6]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

## Prepare train / val / test splits

In [8]:
# Rename label column to 'labels' for transformers
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

#Keep only the columns we actually need
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

# Split train into train/validation
train_val_split = tokenized_datasets["train"].train_test_split(
    test_size=0.2,
    seed=42,
)
train_dataset = train_val_split["train"]
eval_dataset = train_val_split["test"]
test_dataset = tokenized_datasets["test"]

len(train_dataset), len(eval_dataset), len(test_dataset)

(20000, 5000, 25000)

In [9]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## LoRA / PEFT setup

In [10]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_lin", "v_lin"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


## Data collator & metrics

In [16]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")
    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"],
    }

In [12]:
training_args = TrainingArguments(
    output_dir="lora-distilbert-imdb",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

C:\Users\mozog\AppData\Local\Temp\ipykernel_29664\1329056257.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
trainer.train(resume_from_checkpoint=True)

C:\Users\mozog\anaconda3\envs\llm_peft\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1050,0.250400
1100,0.233100
1150,0.212800
1200,0.230700
1250,0.261200
1300,0.267500
1350,0.195700
1400,0.241500
1450,0.180400
1500,0.204900


TrainOutput(global_step=2500, training_loss=0.12630790061950684, metrics={'train_runtime': 15591.1322, 'train_samples_per_second': 2.566, 'train_steps_per_second': 0.16, 'total_flos': 5415353606692608.0, 'train_loss': 0.12630790061950684, 'epoch': 2.0})

In [ ]:
trainer.evaluate(eval_dataset)
trainer.evaluate(test_dataset)

In [19]:
def predict_sentence (text: str):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    pred_id = int(torch.argmax(probs))
    return {
        "text": text,
        "label": id2label[pred_id],
        "confidence": float(probs[pred_id]),
    }

print(predict_sentence("This movie was fantastic, I loved it!"))

print(predict_sentence("The movie had good acting but the story was slow and confusing."))

{'text': 'This movie was fantastic, I loved it!', 'label': 'POSITIVE', 'confidence': 0.9987537860870361}
{'text': 'The movie had good acting but the story was slow and confusing.', 'label': 'NEGATIVE', 'confidence': 0.9858494997024536}
